# MAI204 — EDA Notebook
**Owner:** Arushi Anand — Data & Architecture Lead  
**Branch:** feature/aa-eda  
**Datasets:** FER2013 (folder format) + RAF-DB Basic  

Covers:
1. Mount Drive and clone repo
2. Unzip RAF-DB aligned images
3. FER2013 class distribution and samples
4. RAF-DB class distribution and samples
5. Pooled split sizes
6. Class imbalance analysis and weights
7. README dataset stats summary

## Cell 1 — Mount Drive and Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Clone repo
!git clone https://github.com/devreet-kaur/face-emotion-recognition /content/face-emotion-recognition
%cd /content/face-emotion-recognition

# Switch to your branch
!git checkout feature/aa-eda
!git pull origin feature/aa-eda

print(f'Working directory: {os.getcwd()}')
!ls

In [ ]:
# Install dependencies
!pip install -r requirements.txt -q

## Cell 2 — Copy Datasets from Drive and Unzip RAF-DB

In [ ]:
# ── Adjust these paths to where your datasets live on Drive ──
FER_DRIVE_PATH   = '/content/drive/MyDrive/PATH_TO_YOUR_FER_FOLDER'
RAF_DRIVE_PATH   = '/content/drive/MyDrive/PATH_TO_YOUR_RAF_BASIC_FOLDER'

# Copy datasets into repo data/ folder
!mkdir -p data
!cp -r "{FER_DRIVE_PATH}" data/FER
!cp -r "{RAF_DRIVE_PATH}" data/basic

# Unzip RAF-DB aligned images
!unzip -q data/basic/Image/aligned.zip -d data/basic/Image/

# Verify
print('FER train folders:')
!ls data/FER/train/
print('\nRAF-DB label file (first 3 lines):')
!head -3 data/basic/EmoLabel/list_patition_label.txt
print('\nRAF-DB aligned images (first 5):')
!ls data/basic/Image/aligned/ | head -5

## Cell 3 — Imports and Config

In [ ]:
import cv2
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter

with open('params.yaml') as f:
    P = yaml.safe_load(f)

FER_ROOT  = Path(P['data']['fer2013_path'])   # data/FER
RAF_ROOT  = Path(P['data']['rafdb_path'])     # data/basic
PLOTS_DIR = Path('results/plots')
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

LABELS = {0:'Angry', 1:'Disgust', 2:'Fear', 3:'Happy',
          4:'Neutral', 5:'Sad', 6:'Surprise'}

FER_CLASSES = ['angry','disgust','fear','happy','neutral','sad','surprise']
FER_MAP     = {name: idx for idx, name in enumerate(FER_CLASSES)}

RAF_MAP = {int(k): int(v) for k, v in P['data']['rafdb_label_map'].items()}

print('Config loaded.')
print(f'FER root : {FER_ROOT}')
print(f'RAF root : {RAF_ROOT}')

## Cell 4 — FER2013 Class Distribution

In [ ]:
# Count FER2013 images per class per split
fer_counts = {'train': {}, 'test': {}}

for split in ['train', 'test']:
    split_path = FER_ROOT / split
    for class_dir in sorted(split_path.iterdir()):
        if not class_dir.is_dir(): continue
        name  = class_dir.name.lower()
        label = LABELS[FER_MAP[name]]
        count = len(list(class_dir.glob('*.jpg'))) + len(list(class_dir.glob('*.png')))
        fer_counts[split][label] = count

fer_df = pd.DataFrame(fer_counts).rename_axis('Emotion')
fer_df['total'] = fer_df['train'] + fer_df['test']
fer_df['% of total'] = (fer_df['total'] / fer_df['total'].sum() * 100).round(1)

print('=== FER2013 Class Counts ===')
print(fer_df.to_string())
print(f'\nTotal FER2013 images: {fer_df["total"].sum()}')

happy   = fer_counts['train'].get('Happy', 0)
disgust = fer_counts['train'].get('Disgust', 0)
print(f'Imbalance ratio (Happy/Disgust in train): {happy/disgust:.1f}x')

In [ ]:
# FER2013 class distribution chart
emotions  = list(fer_counts['train'].keys())
train_cnt = list(fer_counts['train'].values())
test_cnt  = list(fer_counts['test'].values())
x = range(len(emotions))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(x, train_cnt, color='#4f46e5', alpha=0.85, label='Train')
axes[0].bar(x, test_cnt,  color='#0891b2', alpha=0.85, label='Test',
            bottom=train_cnt)
axes[0].set_xticks(x)
axes[0].set_xticklabels(emotions, rotation=20, ha='right')
axes[0].set_title('FER2013 — Class Distribution', fontsize=13)
axes[0].set_ylabel('Image Count')
axes[0].legend()

ratios = [fer_counts['train'][e] / disgust for e in emotions]
colors = ['#dc2626' if r > 5 else '#4f46e5' for r in ratios]
axes[1].bar(x, ratios, color=colors, alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(emotions, rotation=20, ha='right')
axes[1].set_title('FER2013 — Imbalance Ratio (vs Disgust)', fontsize=13)
axes[1].set_ylabel('Ratio')
axes[1].axhline(y=1, color='black', linestyle='--', linewidth=0.8)

plt.tight_layout()
out = PLOTS_DIR / 'fer2013_class_distribution.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

In [ ]:
# FER2013 sample images grid
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
fig.suptitle('FER2013 — Sample Images (1 per class)', fontsize=14)

for idx, (folder, label_idx) in enumerate(FER_MAP.items()):
    class_dir = FER_ROOT / 'train' / folder
    imgs = list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
    if not imgs: continue
    img = cv2.imread(str(imgs[0]))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax  = axes.flatten()[idx]
    ax.imshow(img, cmap='gray' if img.shape[2]==1 else None)
    ax.set_title(f'{LABELS[label_idx]}\n({len(imgs)} imgs)', fontsize=9)
    ax.axis('off')

axes.flatten()[7].axis('off')
plt.tight_layout()
out = PLOTS_DIR / 'fer2013_samples.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

## Cell 5 — RAF-DB Class Distribution

In [ ]:
label_file  = RAF_ROOT / 'EmoLabel' / 'list_patition_label.txt'
aligned_dir = RAF_ROOT / 'Image' / 'aligned'

raf_train = Counter()
raf_test  = Counter()
missing   = 0

with open(label_file) as f:
    for line in f:
        line = line.strip()
        if not line: continue
        parts = line.split()
        if len(parts) < 2: continue
        fname, raf_label = parts[0], int(parts[1])
        if raf_label not in RAF_MAP: continue
        unified  = RAF_MAP[raf_label]
        img_path = aligned_dir / fname
        if not img_path.exists():
            missing += 1
            continue
        if fname.startswith('train'):
            raf_train[LABELS[unified]] += 1
        else:
            raf_test[LABELS[unified]] += 1

raf_df = pd.DataFrame({'train': raf_train, 'test': raf_test}).fillna(0).astype(int)
raf_df['total'] = raf_df['train'] + raf_df['test']
raf_df['% of total'] = (raf_df['total'] / raf_df['total'].sum() * 100).round(1)

print('=== RAF-DB Class Counts ===')
print(raf_df.to_string())
print(f'\nTotal RAF-DB images: {raf_df["total"].sum()}')
print(f'Missing aligned images: {missing}')

In [ ]:
# RAF-DB chart
emotions  = list(raf_df.index)
train_cnt = raf_df['train'].tolist()
test_cnt  = raf_df['test'].tolist()
x = range(len(emotions))

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x, train_cnt, color='#059669', alpha=0.85, label='Train')
ax.bar(x, test_cnt,  color='#34d399', alpha=0.85, label='Test', bottom=train_cnt)
ax.set_xticks(x)
ax.set_xticklabels(emotions, rotation=20, ha='right')
ax.set_title('RAF-DB Basic — Class Distribution', fontsize=13)
ax.set_ylabel('Image Count')
ax.legend()
plt.tight_layout()
out = PLOTS_DIR / 'rafdb_class_distribution.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

## Cell 6 — Pooled Split Sizes and Class Weights

In [ ]:
import sys
sys.path.insert(0, '.')
from src.dataset import build_splits, make_weighted_sampler

train_items, val_items, test_items = build_splits()

print(f'\nPooled split sizes:')
print(f'  Train : {len(train_items):>6}')
print(f'  Val   : {len(val_items):>6}')
print(f'  Test  : {len(test_items):>6}')
print(f'  Total : {len(train_items)+len(val_items)+len(test_items):>6}')

fer_n = sum(1 for it in train_items if it[2] == 'fer')
raf_n = sum(1 for it in train_items if it[2] == 'rafdb')
print(f'\nTrain source breakdown:')
print(f'  FER2013: {fer_n}')
print(f'  RAF-DB : {raf_n}')

In [ ]:
# Per-class counts and weights in training set
import numpy as np
from collections import Counter

train_labels = [it[1] for it in train_items]
counts = Counter(train_labels)
total  = len(train_labels)

print('Training set class distribution + weights:')
print(f'{"Emotion":<12} {"Count":>8} {"Weight":>10}')
print('-' * 33)
for i in range(7):
    w = total / (7 * counts.get(i, 1))
    print(f'{LABELS[i]:<12} {counts.get(i,0):>8} {w:>10.4f}')

# Side-by-side bar chart
fer_per  = Counter(it[1] for it in train_items if it[2]=='fer')
raf_per  = Counter(it[1] for it in train_items if it[2]=='rafdb')
emotions = [LABELS[i] for i in range(7)]
fer_vals = [fer_per.get(i,0) for i in range(7)]
raf_vals = [raf_per.get(i,0) for i in range(7)]

x     = np.arange(7)
width = 0.35
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(x-width/2, fer_vals, width, label='FER2013', color='#4f46e5', alpha=0.85)
ax.bar(x+width/2, raf_vals, width, label='RAF-DB',  color='#059669', alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(emotions, rotation=20, ha='right')
ax.set_title('Pooled Train Set — Per-Class Counts by Source', fontsize=13)
ax.set_ylabel('Image Count')
ax.legend()
plt.tight_layout()
out = PLOTS_DIR / 'pooled_train_source_comparison.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved → {out}')

## Cell 7 — README Stats Summary

In [ ]:
fer_total = int(fer_df['total'].sum())
raf_total = int(raf_df['total'].sum())

print('=== README Dataset Statistics ===')
print(f'''
| Dataset   | Images   | Format          | Labels         | Role            |
|-----------|----------|-----------------|----------------|------------------|
| FER2013   | {fer_total:>8} | 48x48 gray JPG  | 1 annotator    | Training pool    |
| RAF-DB    | {raf_total:>8} | 100x100 RGB     | ~40 annotators | Training pool    |
| AffectNet |   ~5,500 | variable RGB    | 1 annotator    | Held-out test    |

Pooled split: {len(train_items)} train / {len(val_items)} val / {len(test_items)} test
Split strategy: Stratified 70/15/15 on label x source
Imbalance fix: WeightedRandomSampler + weighted CrossEntropyLoss
Preprocessing: Resize 224x224, ImageNet normalize
''')
print('EDA complete. Plots saved to:', PLOTS_DIR)

## Cell 8 — Track Datasets with DVC and Push

In [ ]:
# Track both datasets with DVC
!dvc add data/FER data/basic
!git add data/FER.dvc data/basic.dvc .gitignore
!git commit -m "data: track FER and RAF-DB basic with DVC"
!dvc push

In [ ]:
# Commit EDA notebook and plots
!git add notebooks/01_eda.ipynb results/plots/
!git commit -m "feat: EDA notebook complete with FER2013+RAF-DB analysis"
!git push origin feature/aa-eda